In [1]:
import sys
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "results/assemblyworldbench/benchmark/benchmark.json").exists()
)
RESULTS = ROOT / "results"
PAPER = ROOT.parent / "AssemblyWorldBench"
CACHE = ROOT / "notebooks/.cache/paper-analysis"
RECOMPUTE = False
WORKERS = 6
sys.path.insert(0, str(ROOT / "notebooks"))
import paper_analysis as analysis  # noqa: E402

analysis.configure(RESULTS, PAPER, CACHE)

# Benchmark results and paired analyses

All experiment inputs come from the exported `results/` package. No historical run directories or temporary scratchpad are read. The geometry analysis uses the existing keyed evaluator-input interface to resolve dataset-side point clouds and target poses. Derived statistics and caches remain in `notebooks/.cache/paper-analysis/`. PDF figures are written to the sibling paper's `fig/` directories. Run with the project environment (`uv sync --extra episodes --group inspection`).

Intervals are pointwise shape-paired, source-stratified bootstrap intervals; they are not simultaneous significance claims. Missing input records are reported explicitly, never replaced with numbers from a report.

In [2]:
frame = analysis.load_benchmark()
coverage = analysis.audit_available()
display(coverage)

{'version': 'paper-analysis-v2',
 'input_root': 'results',
 'files': 11740,
 'candidate_files': [],
 'missing': {'robustness_pilots': {'status': 'missing',
   'required': 'Per-attempt records for repeated/layout/missing/distractor/observation conditions'},
  'garf_hybrid': {'status': 'missing',
   'required': 'Standard and agent-initialized GARF per-object outputs with checkpoint, seeds and shared protocol'}},
 'garf_diagnostic': 'fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/garf'}

## Source-weighted estimates
PartNet conditions share a shape resample. Each source receives one quarter of the aggregate weight. Costs retain their recorded observation coverage.

In [3]:
import pandas as pd

systems, paired, references = analysis.results_analysis(frame)

display(pd.DataFrame(systems))
display(pd.DataFrame(paired))
display(pd.DataFrame(references))

,system,SR,SR_ci,PA,PA_ci,mean_minutes,cost,cost_n,calls,tokens,scored,archives,runtime_failed,timeouts,other_failed
0,gpt-6-astra,0.59375,"[0.5, 0.68125]",0.808768,"[0.7495243874135733, 0.8629373712225273]",5.828760,2.022897,100,94.01,1.079382,100,100,0,0,0
1,claude-fable-5-1,0.50000,"[0.4125, 0.5875]",0.740704,"[0.6779096644857335, 0.7997524150739611]",22.105088,6.839754,100,161.70,4.603870,100,100,0,0,0
2,claude-opus-5,0.44375,"[0.34375, 0.54375]",0.631079,"[0.550585323530636, 0.7081858857288545]",36.950994,12.377493,88,179.84,16.892775,100,100,12,12,0
3,gpt-5.6-sol,0.11250,"[0.05, 0.175]",0.305962,"[0.23482388358169606, 0.3774820019390332]",8.682869,1.790068,100,208.12,2.769733,100,100,0,0,0
4,claude-sonnet-5,0.07500,"[0.025, 0.125]",0.144167,"[0.09041666666666667, 0.20041666666666663]",20.566276,4.716391,100,195.83,16.451315,100,100,0,0,0
5,gpt-5.6-terra,0.07500,"[0.025, 0.125]",0.111682,"[0.061681547619047615, 0.16220238095238093]",4.001081,0.501779,100,59.87,1.320050,100,100,0,0,0
6,qwen3.8-max-litellm,0.11875,"[0.05625, 0.18125]",0.158586,"[0.09832141122766122, 0.22281662174630923]",54.535726,1.673782,34,107.49,3.673291,100,100,79,65,14
7,deepseek-v4.1-flash,0.00000,"[0.0, 0.0]",0.000000,"[0.0, 0.0]",26.711243,0.162252,78,186.22,2.586482,100,100,51,17,34


,a,b,metric,delta,ci,p_randomization,p_holm
0,gpt-6-astra,claude-fable-5-1,SR,0.093750,"[0.00625, 0.18125]",0.052839,0.581234
1,gpt-6-astra,claude-fable-5-1,PA,0.068064,"[0.012624250315656565, 0.12304455093517591]",0.018760,0.262637
2,gpt-6-astra,claude-opus-5,SR,0.150000,"[0.05625, 0.24375]",0.004250,0.063749
3,gpt-6-astra,claude-opus-5,PA,0.177689,"[0.10509733755827506, 0.254370204402688]",0.000020,0.000560
4,gpt-6-astra,gpt-5.6-sol,SR,0.481250,"[0.38125, 0.58125]",0.000010,0.000560
5,gpt-6-astra,gpt-5.6-sol,PA,0.502806,"[0.4184109882944176, 0.5865203433262645]",0.000010,0.000560
6,gpt-6-astra,claude-sonnet-5,SR,0.518750,"[0.41875, 0.61875]",0.000010,0.000560
7,gpt-6-astra,claude-sonnet-5,PA,0.664601,"[0.5916213256316054, 0.7363383126376547]",0.000010,0.000560
8,gpt-6-astra,gpt-5.6-terra,SR,0.518750,"[0.40625, 0.625]",0.000010,0.000560
9,gpt-6-astra,gpt-5.6-terra,PA,0.697086,"[0.6149236535120581, 0.7747763694410487]",0.000010,0.000560


,system,metric,delta,ci,positive,negative,n
0,gpt-6-astra,PA,0.205945,"[0.09524121711621714, 0.32619978979353975]",12,2,20
1,gpt-6-astra,SR,0.250000,"[0.1, 0.45]",5,0,20
2,claude-fable-5-1,PA,0.193082,"[0.05486947774447775, 0.34338924617049604]",13,5,20
3,claude-fable-5-1,SR,0.200000,"[0.05, 0.4]",4,0,20
4,claude-opus-5,PA,0.156264,"[0.019966491841491865, 0.2996345633533133]",13,4,20
5,claude-opus-5,SR,0.150000,"[0.0, 0.3]",3,0,20
6,gpt-5.6-sol,PA,0.027497,"[-0.051113053613053606, 0.11203401806526804]",6,4,20
7,gpt-5.6-sol,SR,0.000000,"[0.0, 0.0]",0,0,20
8,claude-sonnet-5,PA,-0.038889,"[-0.11666666666666665, 0.0]",0,1,20
9,claude-sonnet-5,SR,0.000000,"[0.0, 0.0]",0,0,20


In [4]:
import paper_figures as figures

analysis.supplementary_results(frame, systems, paired, references)

analyses, missing = figures.load_analyses(frame)
quality = figures.make_quality(frame, analyses)
figures.make_efficiency(systems, quality)
display(pd.DataFrame(analysis.read_json(CACHE / "calibration.json")))

/Users/davidz/Projects/MERL/AssemblyWorldModel/assembly-world-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,system,reports,completed,successes,precision
0,gpt-6-astra,100,74,43,0.581081
1,claude-fable-5-1,100,88,39,0.443182
2,claude-opus-5,88,77,35,0.454545
3,gpt-5.6-sol,100,70,9,0.128571
4,claude-sonnet-5,98,34,5,0.147059
5,gpt-5.6-terra,100,19,5,0.263158
6,qwen3.8-max-litellm,23,9,4,0.444444
7,deepseek-v4.1-flash,55,26,0,0.000000


## Robustness pilot evidence
The exported package is searched for candidate pilot records. No six-object pilot estimates are reported until per-attempt records, variant definitions and scoring identities are present.

In [5]:
display(coverage["missing"]["robustness_pilots"])

{'status': 'missing',
 'required': 'Per-attempt records for repeated/layout/missing/distractor/observation conditions'}

In [6]:
import paper_tables

paper_tables.build(frame)

## Source-level table audit
Recompute each available source-level aggregate from the exported per-object metrics and verify recorded episode checksums. Missing legacy cells remain explicitly unsupported.

In [7]:
source_audit = analysis.audit_source_results()
display(source_audit)

{'verified': [{'source': 'assemblybench/gpt-6-astra/assemblybench-manualbook/evaluation/metrics_summary.json',
   'summary_sha256': 'd9a6ea8b3ce1a3b5a534b86acd1ba87bfa9f97a40d94167b53038e15677c4588',
   'records_sha256': '045fc83ddb07abccc4b065f97a9102e3f1a38473cef477f8e2a7cbab8c75dd63',
   'configured': 280,
   'reported': 279,
   'excluded': ['6772'],
   'episode_checksums_verified': 280,
   'means': {'SCD': 8.541165193514582,
    'PA': 0.7832448220705985,
    'SR': 0.5591397849462365}},
  {'source': 'fantastic-breaks/gpt-6-astra/fantastic-breaks-none/evaluation/chamfer-v2/metrics_summary.json',
   'summary_sha256': 'f93b52458830aa48ca48cced5514801ed1080a00f58ffe6738c06ae1a00970fd',
   'records_sha256': '425e7abee380f4cdd8b95977579ac4bfaecd16ddd3f953990392a8b765f3685e',
   'configured': 150,
   'reported': 150,
   'excluded': [],
   'episode_checksums_verified': 150,
   'means': {'SCD': 1.021944935352916, 'PA': 0.9633333333333334, 'SR': 0.96}},
  {'source': 'fantastic-breaks/gpt-6-as

## Evaluation-set coverage
Generate appendix shape and part counts from results records; repeated reference conditions share shape counts.

In [ ]:
from paper_dataset_coverage import generate
dataset_coverage = generate()


## Free-space SCD upper-tail analysis
Read the five reported PartNet result sets, verify published means, and report the contribution of the largest ceil(5% N) SCD values. Audit initial versus final poses for the Storage/IR tail; do not infer abandonment from PA alone.


In [ ]:
from paper_scd_tail import generate as generate_scd_tail
scd_tail = generate_scd_tail(RESULTS, PAPER, CACHE)
import pandas as pd
display(pd.DataFrame(scd_tail["rows"])[["label", "n", "mean", "median", "k", "tail_share", "remainder_mean"]])
display(pd.DataFrame(scd_tail["cases"]))


## Within-object part-size variation
Full geometry audit from results archives. Vertex-PCA size ratios are invariant to uniform object normalization. Surface-sampling sensitivity and part-count-controlled performance analyses remain pending.

In [1]:
from paper_part_scale_pilot import generate_full
part_scale_report = generate_full()
import pandas as pd
from IPython.display import display
display(pd.DataFrame(part_scale_report["summaries"]))

Analyzed 100/1853 objects


Analyzed 200/1853 objects


Analyzed 300/1853 objects


Analyzed 400/1853 objects


Analyzed 500/1853 objects


...

Analyzed 1600/1853 objects


Analyzed 1700/1853 objects


Analyzed 1800/1853 objects


[
  {
    "dataset": "AssemblyBench",
    "n": 279,
    "q25": 2.2900459997493776,
    "median": 4.3909838699813095,
    "q75": 8.896199365961548,
    "p90": 15.725595066004908,
    "p95": 20.298777846795552,
    "maximum": 48.47311543337784,
    "median_ci95": [
      3.6000659681459077,
      5.535052498478328
    ],
    "fraction_over_10": 0.2007168458781362,
    "fraction_over_20": 0.05734767025089606
  },
  {
    "dataset": "IKEA-Manual",
    "n": 102,
    "q25": 1.6686239578465534,
    "median": 2.202489979162357,
    "q75": 3.2840893276068113,
    "p90": 4.524967695952291,
    "p95": 6.8526425445404975,
    "maximum": 13.3153090393986,
    "median_ci95": [
      2.037461288138938,
      2.7330263619545567
    ],
    "fraction_over_10": 0.0392156862745098,
    "fraction_over_20": 0.0
  },
  {
    "dataset": "PartNet Chair",
    "n": 791,
    "q25": 1.0000000000000013,
    "median": 1.913533875835044,
    "q75": 3.0446699713146197,
    "p90": 6.801276761415075,
    "p95": 11.93567

,dataset,n,q25,median,q75,p90,p95,maximum,median_ci95,fraction_over_10,fraction_over_20
0,AssemblyBench,279,2.290046,4.390984,8.896199,15.725595,20.298778,48.473115,"[3.6000659681459077, 5.535052498478328]",0.200717,0.057348
1,IKEA-Manual,102,1.668624,2.202490,3.284089,4.524968,6.852643,13.315309,"[2.037461288138938, 2.7330263619545567]",0.039216,0.000000
2,PartNet Chair,791,1.000000,1.913534,3.044670,6.801277,11.935679,59.280923,"[1.8124953822664178, 2.0348133520245337]",0.061947,0.021492
3,PartNet Table,533,1.000000,1.388904,3.098350,5.472845,17.474338,64.406922,"[1.1849042843301145, 1.6547770176254506]",0.080675,0.037523
4,PartNet Storage,148,1.000000,1.664676,8.859949,17.681690,28.873794,82.782202,"[1.00000000000002, 2.741257221896757]",0.236486,0.081081
5,PartNet pooled,1472,1.000000,1.789550,3.218559,8.319258,16.373150,82.782202,"[1.6570338962034803, 1.897859375418638]",0.086277,0.033288
